# ModernBERT — CEFR classification (fine-tuning)

Fine-tunes ModernBERT-large on Write & Improve (A2–C1+, 8 levels). One backbone,
two modes set by `cfg.mode`:

- `"nominal"` — cross-entropy over the 8 levels
- `"corn"` — rank-consistent ordinal head + CORN loss

Direct analogue of the T5Gemma 2 encoder-only notebook, swapping the backbone
for ModernBERT. Uses the same `metrics.py`.

**Run both modes** by setting `cfg.mode` to `"corn"` then `"nominal"` (change
`output_dir` accordingly) — the only thing that differs between them is the head
width and the loss, so the QWK/MAE gap is attributable to the ordinal formulation.

In [3]:
import os, json, csv
from dataclasses import dataclass
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments,
    DataCollatorWithPadding,
)

# !pip install coral_pytorch
from coral_pytorch.losses import corn_loss
from coral_pytorch.dataset import corn_label_from_logits
from metrics import evaluate_predictions, LABEL_NAMES


def corn_probas(logits):
    """(N, K-1) CORN logits -> (N, K) class distribution, rows sum to 1."""
    cond = torch.sigmoid(logits)          # P(y > k | y > k-1)
    cum  = torch.cumprod(cond, dim=1)     # P(y > k), k = 0..K-2
    p = torch.zeros(logits.size(0), logits.size(1) + 1)
    p[:, 0]    = 1.0 - cum[:, 0]
    p[:, 1:-1] = cum[:, :-1] - cum[:, 1:]
    p[:, -1]   = cum[:, -1]
    return p.clamp_min(0.0)


LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]
LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)  # 8


@dataclass
class CFG:
    mode: str = "corn"                                # "corn" | "nominal"
    model_id: str = "answerdotai/ModernBERT-large"    # ~395M params
    train_path: str = "data/train.jsonl"
    eval_path: str  = "data/val.jsonl"
    test_path: str  = "data/test.jsonl"
    text_col: str = "text"
    label_col: str = "label"
    max_length: int = 1024
    epochs: float = 4.0
    batch_size: int = 8
    grad_accum: int = 1
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    bf16: bool = True
    use_lora: bool = False
    lora_r: int = 16
    output_dir: str = "runs/modernbert_corn"           # change for nominal
    seed: int = 42
    RUN_TEST_EVAL: bool = False

cfg = CFG()

## Data

In [4]:
def read_rows(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".jsonl":
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)
    elif ext == ".json":
        with open(path, encoding="utf-8") as f:
            yield from json.load(f)
    else:  # csv / tsv
        delim = "\t" if ext == ".tsv" else ","
        with open(path, encoding="utf-8", newline="") as f:
            yield from csv.DictReader(f, delimiter=delim)


class CEFRDataset(Dataset):
    def __init__(self, path, tokenizer, cfg):
        self.ex, skipped = [], 0
        for row in read_rows(path):
            lab = str(row[cfg.label_col]).strip().upper()
            if lab not in LABEL2ID:
                skipped += 1
                continue
            self.ex.append((str(row[cfg.text_col]), LABEL2ID[lab]))
        print(f"{path}: kept {len(self.ex)}, skipped {skipped} out-of-scope")
        self.tok, self.max_length = tokenizer, cfg.max_length

    def __len__(self):
        return len(self.ex)

    def __getitem__(self, i):
        text, label = self.ex[i]
        enc = self.tok(text, truncation=True, max_length=self.max_length)
        enc["labels"] = label
        return enc

In [5]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
train_ds = CEFRDataset(cfg.train_path, tokenizer, cfg)
eval_ds  = CEFRDataset(cfg.eval_path,  tokenizer, cfg)
collator = DataCollatorWithPadding(tokenizer)

data/train.jsonl: kept 3797, skipped 0 out-of-scope
data/val.jsonl: kept 599, skipped 0 out-of-scope


## Model

Mean-pool encoder outputs (attention-mask aware), then project through
a single linear head whose width depends on mode:
- nominal: `hidden → 8`
- corn: `hidden → 7` (K−1 rank-consistent logits)

In [6]:
from transformers.modeling_outputs import SequenceClassifierOutput


class ModernBERTOrdinalClassifier(nn.Module):
    def __init__(self, model_id, num_classes=8, mode="corn", dropout=0.1, dtype=None):
        super().__init__()
        assert mode in {"corn", "nominal"}
        self.mode = mode
        self.encoder = AutoModel.from_pretrained(model_id, torch_dtype=dtype)
        hidden = self.encoder.config.hidden_size
        out_dim = num_classes - 1 if mode == "corn" else num_classes
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, out_dim)
        self.head.to(torch.float32)  # stable logits even with bf16 backbone

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = out.last_hidden_state
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).to(h.dtype)
            pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        else:
            pooled = h.mean(1)
        logits = self.head(self.dropout(pooled).to(torch.float32))
        return SequenceClassifierOutput(logits=logits)

In [7]:
dtype = torch.bfloat16 if cfg.bf16 else torch.float32
model = ModernBERTOrdinalClassifier(
    cfg.model_id, num_classes=NUM_CLASSES, mode=cfg.mode, dtype=dtype,
)

if cfg.use_lora:
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=cfg.lora_r, lora_alpha=2 * cfg.lora_r, lora_dropout=0.05,
        target_modules=["Wqkv"],       # ModernBERT fuses QKV into one projection
        modules_to_save=["head"],
    ))
    model.print_trainable_parameters()

total = sum(p.numel() for p in model.parameters())
train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"params: {total:,} total, {train:,} trainable")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


params: 394,788,871 total, 394,788,871 trainable


## Trainer

In [8]:
class OrdinalTrainer(Trainer):
    def __init__(self, *a, mode="corn", **k):
        super().__init__(*a, **k)
        self.mode = mode

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        logits = out.logits
        if self.mode == "corn":
            loss = corn_loss(logits, labels.long(), NUM_CLASSES)
        else:
            loss = torch.nn.functional.cross_entropy(logits, labels.long())
        return (loss, out) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        labels = inputs.pop("labels")
        with torch.no_grad():
            out = model(**inputs)
            logits = out.logits
            if self.mode == "corn":
                loss = corn_loss(logits, labels.long(), NUM_CLASSES)
            else:
                loss = torch.nn.functional.cross_entropy(logits, labels.long())
        if prediction_loss_only:
            return (loss, None, None)
        return (loss, logits, labels)

def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = torch.as_tensor(np.asarray(logits, dtype=np.float32))
    if cfg.mode == "corn":
        preds = corn_label_from_logits(logits).numpy()
    else:
        preds = logits.argmax(-1).numpy()
    return evaluate_predictions(eval_pred.label_ids, preds)

In [9]:
args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    weight_decay=cfg.weight_decay,
    warmup_ratio=cfg.warmup_ratio,
    bf16=cfg.bf16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_qwk",
    greater_is_better=True,
    save_total_limit=2,
    seed=cfg.seed,
    report_to="none",
)

trainer = OrdinalTrainer(
    model=model, args=args, mode=cfg.mode,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, compute_metrics=compute_metrics,
)
trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,0.329066,0.362440,0.826251,0.659432,0.469115,0.893155,0.295986,0.348956,0.296270,0.441408,0.469115,0.430563
2,0.293879,0.335343,0.838418,0.604341,0.487479,0.921536,0.357488,0.346997,0.323317,0.471641,0.487479,0.461530
3,0.293724,0.324378,0.848994,0.595993,0.487479,0.929883,0.346336,0.351110,0.338608,0.462580,0.487479,0.462908
4,0.283608,0.322409,0.846568,0.592654,0.490818,0.931553,0.350032,0.346729,0.334934,0.464101,0.490818,0.463443


TrainOutput(global_step=1900, training_loss=0.3308796631662469, metrics={'train_runtime': 175.5917, 'train_samples_per_second': 86.496, 'train_steps_per_second': 10.821, 'total_flos': 0.0, 'train_loss': 0.3308796631662469, 'epoch': 4.0})

## Dev evaluation — metrics + predictions

In [ ]:
dev_results = trainer.evaluate()
print("best dev:", json.dumps(dev_results, indent=2))

# --- extract predictions ---
dev_out = trainer.predict(eval_ds)
dev_logits = dev_out.predictions
if isinstance(dev_logits, tuple):
    dev_logits = dev_logits[0]
dev_logits_t = torch.as_tensor(np.asarray(dev_logits, dtype=np.float32))
if cfg.mode == "corn":
    dev_preds = corn_label_from_logits(dev_logits_t).numpy()
else:
    dev_preds = dev_logits_t.argmax(-1).numpy()

dev_gold = dev_out.label_ids

# --- probabilities ---
print("logits shape:", tuple(dev_logits_t.shape))   # (N, 7) corn | (N, 8) nominal
probs = (corn_probas(dev_logits_t) if cfg.mode == "corn"
         else torch.softmax(dev_logits_t, dim=-1)).numpy()

srt = np.sort(probs, axis=1)[:, ::-1]
rows = []
for i, (g, p) in enumerate(zip(dev_gold, dev_preds)):
    g, p = int(g), int(p)
    lo, hi = max(0, p - 1), min(NUM_CLASSES - 1, p + 1)
    rows.append(dict(
        gold=g, pred=p, gold_label=LEVELS[g], pred_label=LEVELS[p],
        correct=int(g == p), adjacent=int(abs(g - p) <= 1),
        p_pred=float(probs[i, p]),
        p_adjacent=float(probs[i, lo:hi + 1].sum()),
        margin=float(srt[i, 0] - srt[i, 1]),
        entropy=float(-(probs[i] * np.log(probs[i] + 1e-12)).sum()),
        exp_level=float((probs[i] * np.arange(NUM_CLASSES)).sum()),
        **{f"p_{lvl}": float(probs[i, j]) for j, lvl in enumerate(LEVELS)},
    ))

import pandas as pd
dev_df = pd.DataFrame(rows)

# --- save ---
os.makedirs(cfg.output_dir, exist_ok=True)
with open(os.path.join(cfg.output_dir, "dev_metrics.json"), "w") as f:
    json.dump(dev_results, f, indent=2)
dev_df.to_csv(os.path.join(cfg.output_dir, "dev_predictions.csv"), index=False)

print(f"\nSaved to {cfg.output_dir}/dev_metrics.json and dev_predictions.csv")
print(f"prob rows sum to 1: {np.allclose(probs.sum(1), 1.0, atol=1e-4)}")

Training Loss,Validation Loss,Epoch,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
0.283608,0.324378,4,0.848994,0.595993,0.487479,0.929883,0.346336,0.351110,0.338608,0.462580,0.487479,0.462908


best dev: {
  "eval_loss": 0.32437846064567566,
  "eval_qwk": 0.8489935014233125,
  "eval_mae": 0.5959933222036727,
  "eval_accuracy": 0.48747913188647746,
  "eval_adjacent_accuracy": 0.9298831385642737,
  "eval_precision_macro": 0.34633586336365196,
  "eval_recall_macro": 0.35110950481681424,
  "eval_f1_macro": 0.33860825992773425,
  "eval_precision_weighted": 0.4625795670833163,
  "eval_recall_weighted": 0.48747913188647746,
  "eval_f1_weighted": 0.4629080675613424
}


logits shape: (599, 7)

Saved to runs/modernbert_corn/dev_metrics.json and dev_predictions.csv
prob rows sum to 1: True


In [11]:
# --- mode-collapse check ---
print("dev pred distribution:", Counter(dev_preds.tolist()))
print("dev gold distribution:", Counter(dev_gold.tolist()))

dev pred distribution: Counter({1: 182, 4: 126, 3: 124, 2: 83, 5: 63, 6: 21})
dev gold distribution: Counter({1: 136, 2: 128, 4: 110, 3: 95, 5: 49, 6: 37, 0: 30, 7: 14})


## Confusion matrix

In [12]:
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

counts = confusion_matrix(dev_gold, dev_preds, labels=list(range(NUM_CLASSES)))
row_sums = counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = np.divide(counts, row_sums, where=row_sums != 0)
norm = np.nan_to_num(norm)

annot = np.empty_like(counts, dtype=object)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        annot[i, j] = f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"

fig = go.Figure(go.Heatmap(
    z=norm, x=LEVELS, y=LEVELS,
    text=annot, texttemplate="%{text}",
    hoverongaps=False,
    colorscale="Greys", zmin=0, zmax=1,
    colorbar=dict(title="Row-norm"),
))
fig.update_layout(
    title=f"ModernBERT-large {cfg.mode} — dev confusion matrix",
    xaxis=dict(title="Predicted"),
    yaxis=dict(title="True", autorange="reversed"),
    font=dict(family="Arial", size=14, color="black"),
    width=600, height=550,
)
fig.show()

## Predicted-label distribution (ridgeline)

In [13]:
import plotly.graph_objects as go

def level_proportions(labels, n_levels=NUM_CLASSES):
    counts = np.bincount(labels, minlength=n_levels).astype(float)
    total = counts.sum()
    return counts / total if total else counts

ridge_rows = [
    ("True labels", dev_gold),
    (f"ModernBERT {cfg.mode}", dev_preds),
]

def row_color(name):
    if "True" in name:  return "rgba(90,90,90,0.55)"
    if "corn" in name:  return "rgba(200,90,80,0.55)"
    return "rgba(70,110,170,0.55)"

spacing = 1.0
scale   = 1.3
n_rows  = len(ridge_rows)

fig = go.Figure()
for i, (name, labels) in enumerate(ridge_rows):
    base = (n_rows - 1 - i) * spacing
    p = level_proportions(labels)

    xs, ys = [], []
    for j in range(NUM_CLASSES):
        xs += [j - 0.5, j + 0.5]
        ys += [base + scale * p[j]] * 2
    poly_x = [-0.5] + xs + [NUM_CLASSES - 0.5]
    poly_y = [base] + ys + [base]

    fig.add_trace(go.Scatter(
        x=poly_x, y=poly_y, fill="toself", mode="lines",
        line=dict(color="black", width=1), fillcolor=row_color(name),
        hoverinfo="skip", showlegend=False,
    ))

    mean_idx = float(np.dot(np.arange(NUM_CLASSES), p))
    fig.add_trace(go.Scatter(
        x=[mean_idx], y=[base], mode="markers",
        marker=dict(color="black", size=10, symbol="line-ns-open"),
        hoverinfo="skip", showlegend=False,
    ))

true_mean = float(np.dot(np.arange(NUM_CLASSES), level_proportions(dev_gold)))
fig.add_vline(x=true_mean, line=dict(color="black", width=1, dash="dash"))

fig.update_layout(
    title=f"Predicted CEFR-level distribution — ModernBERT-large {cfg.mode}",
    xaxis=dict(tickmode="array", tickvals=list(range(NUM_CLASSES)), ticktext=LEVELS,
               title="CEFR level", range=[-0.6, NUM_CLASSES - 0.4]),
    yaxis=dict(tickmode="array",
               tickvals=[(n_rows - 1 - i) * spacing for i in range(n_rows)],
               ticktext=[name for name, _ in ridge_rows],
               showgrid=False, zeroline=False),
    height=105 * n_rows + 120, width=900,
    font=dict(family="Arial", size=14, color="black"),
    plot_bgcolor="white",
)
fig.show()

# Does confidence separate correct from wrong?
### 0.5 is chance, >0.7 is what I want to see

In [17]:
import pandas as pd
from sklearn.metrics import roc_auc_score

df = pd.read_csv(os.path.join(cfg.output_dir, "dev_predictions.csv"))

for col, s in [("p_pred", df["p_pred"]), ("margin", df["margin"]),
               ("entropy", -df["entropy"]), ("p_adjacent", df["p_adjacent"])]:
    print(f"{col:12s} exact {roc_auc_score(df['correct'], s):.3f}   "
          f"adjacent {roc_auc_score(df['adjacent'], s):.3f}")

print()
print(df.groupby("correct")[["p_pred", "margin", "entropy"]].mean().round(3))

p_pred       exact 0.614   adjacent 0.728
margin       exact 0.598   adjacent 0.663
entropy      exact 0.602   adjacent 0.735
p_adjacent   exact 0.616   adjacent 0.741

         p_pred  margin  entropy
correct                         
0         0.454   0.162    1.314
1         0.487   0.200    1.260


In [20]:
import plotly.graph_objects as go

fig = go.Figure()
for lab, name, col in [(1, "Correct", "rgba(70,110,170,0.6)"),
                       (0, "Wrong", "rgba(200,90,80,0.6)")]:
    fig.add_histogram(x=df.loc[df["correct"] == lab, "p_pred"], name=name,
                      marker_color=col, xbins=dict(start=0, end=1, size=0.025),
                      histnorm="probability")
fig.update_layout(
    barmode="overlay",
    title=dict(text="Confidence in the predicted label, by outcome"),
    xaxis_title="p_pred", yaxis_title="Proportion of cases",
    font=dict(family="Arial", size=16, color="black"),
    plot_bgcolor="white", width=780, height=440,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.show()

In [21]:
b = pd.cut(df["p_pred"], [0, .3, .4, .5, .6, .7, 1.0])
print(df.groupby(b, observed=True).agg(
    n=("correct", "size"),
    exact=("correct", "mean"),
    adjacent=("adjacent", "mean"),
    mean_conf=("p_pred", "mean"),
).round(3))

              n  exact  adjacent  mean_conf
p_pred                                     
(0.0, 0.3]   17  0.118     0.588      0.279
(0.3, 0.4]   99  0.323     0.859      0.357
(0.4, 0.5]  248  0.484     0.944      0.452
(0.5, 0.6]  213  0.606     0.977      0.542
(0.6, 0.7]   22  0.409     0.909      0.636


# Risk coverage curve

In [18]:
d = df.sort_values("p_pred", ascending=False).reset_index(drop=True)
d["coverage"] = (d.index + 1) / len(d)
d["cum_acc"] = d["correct"].expanding().mean()
d["cum_adj"] = d["adjacent"].expanding().mean()
for c in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    r = d.iloc[int(c * len(d)) - 1]
    print(f"coverage {c:.0%}  threshold {r['p_pred']:.3f}  "
          f"acc {r['cum_acc']:.3f}  adjacent {r['cum_adj']:.3f}")

coverage 50%  threshold 0.472  acc 0.569  adjacent 0.970
coverage 60%  threshold 0.452  acc 0.563  adjacent 0.961
coverage 70%  threshold 0.433  acc 0.549  adjacent 0.964
coverage 80%  threshold 0.403  acc 0.534  adjacent 0.956
coverage 90%  threshold 0.354  acc 0.512  adjacent 0.948
coverage 100%  threshold 0.246  acc 0.487  adjacent 0.930


In [19]:
import plotly.graph_objects as go

d = df.sort_values("p_pred", ascending=False).reset_index(drop=True)
d["coverage"] = (np.arange(len(d)) + 1) / len(d)
d["cum_acc"] = d["correct"].expanding().mean()
d["cum_adj"] = d["adjacent"].expanding().mean()
d = d[d["coverage"] >= 0.05]

fig = go.Figure()
fig.add_scatter(x=d["coverage"], y=d["cum_adj"], name="Adjacent accuracy",
                line=dict(color="rgb(70,110,170)", width=2))
fig.add_scatter(x=d["coverage"], y=d["cum_acc"], name="Exact accuracy",
                line=dict(color="rgb(200,90,80)", width=2))
fig.add_hline(y=d["cum_adj"].iloc[-1], line=dict(color="rgb(70,110,170)", width=1, dash="dot"))
fig.add_hline(y=d["cum_acc"].iloc[-1], line=dict(color="rgb(200,90,80)", width=1, dash="dot"))
fig.update_layout(
    title=dict(text="Risk–coverage: ModernBERT CORN (val)"),
    xaxis_title="Coverage (proportion auto-scored, most confident first)",
    yaxis_title="Accuracy on retained cases",
    font=dict(family="Arial", size=16, color="black"),
    plot_bgcolor="white", width=780, height=480,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.update_xaxes(tickformat=".0%", gridcolor="rgba(0,0,0,0.1)")
fig.update_yaxes(range=[0.4, 1.0], gridcolor="rgba(0,0,0,0.1)")
fig.show()

## Test evaluation (guarded)

In [ ]:
# if cfg.RUN_TEST_EVAL:
#     test_ds = CEFRDataset(cfg.test_path, tokenizer, cfg)
#     test_out = trainer.predict(test_ds)
#     print("test:", json.dumps(test_out.metrics, indent=2))
#     with open(os.path.join(cfg.output_dir, "test_metrics.json"), "w") as f:
#         json.dump(test_out.metrics, f, indent=2)
# else:
#     print("Test eval skipped (RUN_TEST_EVAL = False)")

# Nominal classifier

In [22]:
import gc
del trainer, model
gc.collect(); torch.cuda.empty_cache()

cfg.mode       = "nominal"
cfg.output_dir = "runs/modernbert_nominal"

In [23]:
dtype = torch.bfloat16 if cfg.bf16 else torch.float32
model = ModernBERTOrdinalClassifier(
    cfg.model_id, num_classes=NUM_CLASSES, mode=cfg.mode, dtype=dtype,
)

args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    weight_decay=cfg.weight_decay,
    warmup_ratio=cfg.warmup_ratio,
    bf16=cfg.bf16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_qwk",
    greater_is_better=True,
    save_total_limit=2,
    seed=cfg.seed,
    report_to="none",
)

trainer = OrdinalTrainer(
    model=model, args=args, mode=cfg.mode,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, compute_metrics=compute_metrics,
)
trainer.train()

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,1.405770,1.321957,0.827374,0.652755,0.472454,0.891486,0.327585,0.331659,0.295245,0.455435,0.472454,0.418964
2,1.207062,1.239063,0.851046,0.585977,0.497496,0.928214,0.353659,0.372255,0.347180,0.473566,0.497496,0.469477
3,1.174227,1.184471,0.857923,0.562604,0.519199,0.933222,0.375676,0.390776,0.373034,0.490022,0.519199,0.488108
4,1.156032,1.177921,0.857945,0.564274,0.515860,0.933222,0.366416,0.380808,0.363687,0.483680,0.515860,0.484899


TrainOutput(global_step=1900, training_loss=1.349552536010742, metrics={'train_runtime': 179.6425, 'train_samples_per_second': 84.546, 'train_steps_per_second': 10.577, 'total_flos': 0.0, 'train_loss': 1.349552536010742, 'epoch': 4.0})

In [36]:
dev_results = trainer.evaluate()
print("best dev:", json.dumps(dev_results, indent=2))

# --- extract predictions ---
dev_out = trainer.predict(eval_ds)
dev_logits = dev_out.predictions
if isinstance(dev_logits, tuple):
    dev_logits = dev_logits[0]
dev_logits_t = torch.as_tensor(np.asarray(dev_logits, dtype=np.float32))
if cfg.mode == "corn":
    dev_preds = corn_label_from_logits(dev_logits_t).numpy()
else:
    dev_preds = dev_logits_t.argmax(-1).numpy()

dev_gold = dev_out.label_ids

# --- probabilities ---
probs = (corn_probas(dev_logits_t) if cfg.mode == "corn"
         else torch.softmax(dev_logits_t, dim=-1)).numpy()

# --- save ---
os.makedirs(cfg.output_dir, exist_ok=True)
with open(os.path.join(cfg.output_dir, "dev_metrics.json"), "w") as f:
    json.dump(dev_results, f, indent=2)
with open(os.path.join(cfg.output_dir, "dev_predictions.csv"), "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["gold", "pred", "gold_label", "pred_label", "p_pred"])
    for i, (g, p) in enumerate(zip(dev_gold, dev_preds)):
        w.writerow([int(g), int(p), LEVELS[int(g)], LEVELS[int(p)], float(probs[i, int(p)])])

print(f"\nSaved to {cfg.output_dir}/dev_metrics.json and dev_predictions.csv")
print(f"mode={cfg.mode}  logits {tuple(dev_logits_t.shape)}  "
      f"rows sum to 1: {np.allclose(probs.sum(1), 1.0, atol=1e-4)}")

Training Loss,Validation Loss,Epoch,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1.156032,1.177921,4,0.857945,0.564274,0.515860,0.933222,0.366416,0.380808,0.363687,0.483680,0.515860,0.484899


best dev: {
  "eval_loss": 1.1779212951660156,
  "eval_qwk": 0.8579454590481148,
  "eval_mae": 0.5642737896494157,
  "eval_accuracy": 0.5158597662771286,
  "eval_adjacent_accuracy": 0.9332220367278798,
  "eval_precision_macro": 0.36641628804340665,
  "eval_recall_macro": 0.3808084360874021,
  "eval_f1_macro": 0.3636865205801947,
  "eval_precision_weighted": 0.4836803779238911,
  "eval_recall_weighted": 0.5158597662771286,
  "eval_f1_weighted": 0.4848989998788171
}



Saved to runs/modernbert_nominal/dev_metrics.json and dev_predictions.csv
mode=nominal  logits (599, 8)  rows sum to 1: True


In [37]:
# --- mode-collapse check ---
print("dev pred distribution:", Counter(dev_preds.tolist()))
print("dev gold distribution:", Counter(dev_gold.tolist()))

dev pred distribution: Counter({1: 198, 3: 125, 4: 118, 2: 77, 5: 55, 6: 26})
dev gold distribution: Counter({1: 136, 2: 128, 4: 110, 3: 95, 5: 49, 6: 37, 0: 30, 7: 14})


In [38]:
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

counts = confusion_matrix(dev_gold, dev_preds, labels=list(range(NUM_CLASSES)))
row_sums = counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = np.divide(counts, row_sums, where=row_sums != 0)
norm = np.nan_to_num(norm)

annot = np.empty_like(counts, dtype=object)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        annot[i, j] = f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"

fig = go.Figure(go.Heatmap(
    z=norm, x=LEVELS, y=LEVELS,
    text=annot, texttemplate="%{text}",
    hoverongaps=False,
    colorscale="Greys", zmin=0, zmax=1,
    colorbar=dict(title="Row-norm"),
))
fig.update_layout(
    title=f"ModernBERT-large {cfg.mode} — dev confusion matrix",
    xaxis=dict(title="Predicted"),
    yaxis=dict(title="True", autorange="reversed"),
    font=dict(family="Arial", size=14, color="black"),
    width=600, height=550,
)
fig.show()

In [39]:
import plotly.graph_objects as go

def level_proportions(labels, n_levels=NUM_CLASSES):
    counts = np.bincount(labels, minlength=n_levels).astype(float)
    total = counts.sum()
    return counts / total if total else counts

ridge_rows = [
    ("True labels", dev_gold),
    (f"ModernBERT {cfg.mode}", dev_preds),
]

def row_color(name):
    if "True" in name:  return "rgba(90,90,90,0.55)"
    if "corn" in name:  return "rgba(200,90,80,0.55)"
    return "rgba(70,110,170,0.55)"

spacing = 1.0
scale   = 1.3
n_rows  = len(ridge_rows)

fig = go.Figure()
for i, (name, labels) in enumerate(ridge_rows):
    base = (n_rows - 1 - i) * spacing
    p = level_proportions(labels)

    xs, ys = [], []
    for j in range(NUM_CLASSES):
        xs += [j - 0.5, j + 0.5]
        ys += [base + scale * p[j]] * 2
    poly_x = [-0.5] + xs + [NUM_CLASSES - 0.5]
    poly_y = [base] + ys + [base]

    fig.add_trace(go.Scatter(
        x=poly_x, y=poly_y, fill="toself", mode="lines",
        line=dict(color="black", width=1), fillcolor=row_color(name),
        hoverinfo="skip", showlegend=False,
    ))

    mean_idx = float(np.dot(np.arange(NUM_CLASSES), p))
    fig.add_trace(go.Scatter(
        x=[mean_idx], y=[base], mode="markers",
        marker=dict(color="black", size=10, symbol="line-ns-open"),
        hoverinfo="skip", showlegend=False,
    ))

true_mean = float(np.dot(np.arange(NUM_CLASSES), level_proportions(dev_gold)))
fig.add_vline(x=true_mean, line=dict(color="black", width=1, dash="dash"))

fig.update_layout(
    title=f"Predicted CEFR-level distribution — ModernBERT-large {cfg.mode}",
    xaxis=dict(tickmode="array", tickvals=list(range(NUM_CLASSES)), ticktext=LEVELS,
               title="CEFR level", range=[-0.6, NUM_CLASSES - 0.4]),
    yaxis=dict(tickmode="array",
               tickvals=[(n_rows - 1 - i) * spacing for i in range(n_rows)],
               ticktext=[name for name, _ in ridge_rows],
               showgrid=False, zeroline=False),
    height=105 * n_rows + 120, width=900,
    font=dict(family="Arial", size=14, color="black"),
    plot_bgcolor="white",
)
fig.show()

---

## Cross-condition comparison (run after both CORN and nominal)

Load saved `dev_predictions.csv` from both output dirs and plot
side-by-side confusion matrices + combined ridgeline.

In [40]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

CORN_DIR    = "runs/modernbert_corn"
NOMINAL_DIR = "runs/modernbert_nominal"

conditions = {}
for label, d in [("corn", CORN_DIR), ("nominal", NOMINAL_DIR)]:
    path = os.path.join(d, "dev_predictions.csv")
    if os.path.exists(path):
        conditions[label] = pd.read_csv(path)
        print(f"Loaded {label}: {len(conditions[label])} rows")
    else:
        print(f"{path} not found — run {label} condition first")

if len(conditions) < 2:
    print("\nNeed both conditions to compare. Skipping.")

Loaded corn: 599 rows
Loaded nominal: 599 rows


In [41]:
# --- side-by-side confusion matrices ---
if len(conditions) == 2:
    def _annot(counts, norm):
        a = np.empty_like(counts, dtype=object)
        for i in range(counts.shape[0]):
            for j in range(counts.shape[1]):
                a[i, j] = f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"
        return a

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["CORN", "Nominal"],
        horizontal_spacing=0.10,
    )
    fig.update_layout(coloraxis=dict(
        colorscale="Greys", cmin=0, cmax=1,
        colorbar=dict(title="Row-norm"),
    ))

    for k, (label, df) in enumerate(conditions.items()):
        c = k + 1
        cm = confusion_matrix(df["gold"], df["pred"], labels=list(range(NUM_CLASSES)))
        rs = cm.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            nm = np.divide(cm, rs, where=rs != 0)
        nm = np.nan_to_num(nm)
        fig.add_trace(
            go.Heatmap(
                z=nm, x=LEVELS, y=LEVELS,
                text=_annot(cm, nm), texttemplate="%{text}",
                hoverongaps=False, coloraxis="coloraxis",
            ),
            row=1, col=c,
        )
        fig.update_xaxes(title_text="Predicted", row=1, col=c)
        fig.update_yaxes(title_text="True", autorange="reversed", row=1, col=c)

    fig.update_layout(
        title="ModernBERT-large — CORN vs Nominal (dev)",
        font=dict(family="Arial", size=14, color="black"),
        width=1100, height=520,
        margin=dict(l=80, r=100, t=100, b=80),
    )
    fig.show()

In [42]:
# --- combined ridgeline ---
if len(conditions) == 2:
    def level_props(series, n=NUM_CLASSES):
        c = np.bincount(series.to_numpy(), minlength=n).astype(float)
        return c / c.sum() if c.sum() else c

    ref_gold = conditions["corn"]["gold"]  # same gold in both
    ridge_rows = [
        ("True labels",           ref_gold,                      "rgba(90,90,90,0.55)"),
        ("ModernBERT CORN",       conditions["corn"]["pred"],    "rgba(200,90,80,0.55)"),
        ("ModernBERT Nominal",    conditions["nominal"]["pred"], "rgba(70,110,170,0.55)"),
    ]

    spacing = 1.0
    scale   = 1.3
    n_rows  = len(ridge_rows)

    fig = go.Figure()
    for i, (name, series, color) in enumerate(ridge_rows):
        base = (n_rows - 1 - i) * spacing
        p = level_props(series)

        xs, ys = [], []
        for j in range(NUM_CLASSES):
            xs += [j - 0.5, j + 0.5]
            ys += [base + scale * p[j]] * 2
        poly_x = [-0.5] + xs + [NUM_CLASSES - 0.5]
        poly_y = [base] + ys + [base]

        fig.add_trace(go.Scatter(
            x=poly_x, y=poly_y, fill="toself", mode="lines",
            line=dict(color="black", width=1), fillcolor=color,
            hoverinfo="skip", showlegend=False,
        ))
        mean_idx = float(np.dot(np.arange(NUM_CLASSES), p))
        fig.add_trace(go.Scatter(
            x=[mean_idx], y=[base], mode="markers",
            marker=dict(color="black", size=10, symbol="line-ns-open"),
            hoverinfo="skip", showlegend=False,
        ))

    true_mean = float(np.dot(np.arange(NUM_CLASSES), level_props(ref_gold)))
    fig.add_vline(x=true_mean, line=dict(color="black", width=1, dash="dash"))

    fig.update_layout(
        title="Predicted CEFR-level distributions — ModernBERT-large CORN vs Nominal",
        xaxis=dict(tickmode="array", tickvals=list(range(NUM_CLASSES)), ticktext=LEVELS,
                   title="CEFR level", range=[-0.6, NUM_CLASSES - 0.4]),
        yaxis=dict(tickmode="array",
                   tickvals=[(n_rows - 1 - i) * spacing for i in range(n_rows)],
                   ticktext=[name for name, _, _ in ridge_rows],
                   showgrid=False, zeroline=False),
        height=105 * n_rows + 120, width=900,
        font=dict(family="Arial", size=14, color="black"),
        plot_bgcolor="white",
    )
    fig.show()

In [43]:
# Cell: summary table of dev metrics — CORN vs nominal
import os, json
import pandas as pd

conditions = [
    ("CORN",    "runs/modernbert_corn"),
    ("Nominal", "runs/modernbert_nominal"),
]

drop = {"loss", "runtime", "samples_per_second", "steps_per_second", "epoch"}

cols = {}
for name, run_dir in conditions:
    with open(os.path.join(run_dir, "dev_metrics.json")) as f:
        m = json.load(f)
    cols[name] = {
        (k[5:] if k.startswith("eval_") else k): v
        for k, v in m.items()
        if (k[5:] if k.startswith("eval_") else k) not in drop
    }

table = pd.DataFrame(cols)

preferred = ["qwk", "mae", "exact_acc", "adjacent_acc",
             "f1_macro", "f1_weighted",
             "precision_macro", "recall_macro",
             "precision_weighted", "recall_weighted"]
order = [m for m in preferred if m in table.index] + \
        [m for m in table.index if m not in preferred]
table = table.loc[order].round(2)

table.to_csv("runs/modernbert_dev_summary.csv")
table

,CORN,Nominal
qwk,0.85,0.86
mae,0.60,0.56
f1_macro,0.34,0.36
f1_weighted,0.46,0.48
precision_macro,0.35,0.37
recall_macro,0.35,0.38
precision_weighted,0.46,0.48
recall_weighted,0.49,0.52
accuracy,0.49,0.52
adjacent_accuracy,0.93,0.93


# AUROC / probabilities

In [44]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

conds = [("CORN", "runs/modernbert_corn"), ("Nominal", "runs/modernbert_nominal")]

def load_preds(run_dir):
    d = pd.read_csv(f"{run_dir}/dev_predictions.csv")
    d["correct"] = (d["gold"] == d["pred"]).astype(int)
    d["adjacent"] = (d["gold"] - d["pred"]).abs().le(1).astype(int)
    return d

def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    gap = (g["correct"].mean() - g["p_pred"].mean()).abs()   # abs per bin
    return float((gap * g.size()).sum() / len(d))

rows = []
for name, run_dir in conds:
    d = load_preds(run_dir)
    if "p_pred" not in d.columns:
        print(f"{name}: no p_pred — re-run the prediction cell for this condition")
        continue
    rows.append(dict(
        condition=name,
        exact=d["correct"].mean(),
        adjacent=d["adjacent"].mean(),
        mean_conf=d["p_pred"].mean(),
        conf_correct=d.loc[d["correct"] == 1, "p_pred"].mean(),
        conf_wrong=d.loc[d["correct"] == 0, "p_pred"].mean(),
        auroc_exact=roc_auc_score(d["correct"], d["p_pred"]),
        auroc_adj=roc_auc_score(d["adjacent"], d["p_pred"]),
        ece=ece(d),
    ))

print(pd.DataFrame(rows).round(3).to_string(index=False))

condition  exact  adjacent  mean_conf  conf_correct  conf_wrong  auroc_exact  auroc_adj   ece
     CORN  0.487     0.930      0.470         0.487       0.454        0.614      0.728 0.054
  Nominal  0.516     0.933      0.485         0.496       0.473        0.588      0.687 0.038


In [46]:
for name, run_dir in conds:
    d = load_preds(run_dir)          # was pd.read_csv(...)
    b = pd.cut(d["p_pred"], [0, .3, .4, .5, .6, .7, 1.0])
    print(f"\n{name}")
    print(d.groupby(b, observed=True).agg(
        n=("correct", "size"), exact=("correct", "mean"),
        adjacent=("adjacent", "mean"), mean_conf=("p_pred", "mean")).round(3))


CORN
              n  exact  adjacent  mean_conf
p_pred                                     
(0.0, 0.3]   17  0.118     0.588      0.279
(0.3, 0.4]   99  0.323     0.859      0.357
(0.4, 0.5]  248  0.484     0.944      0.452
(0.5, 0.6]  213  0.606     0.977      0.542
(0.6, 0.7]   22  0.409     0.909      0.636

Nominal
              n  exact  adjacent  mean_conf
p_pred                                     
(0.0, 0.3]    4  0.250     0.750      0.287
(0.3, 0.4]   87  0.402     0.862      0.362
(0.4, 0.5]  234  0.483     0.923      0.451
(0.5, 0.6]  240  0.583     0.971      0.544
(0.6, 0.7]   33  0.576     0.939      0.637
(0.7, 1.0]    1  1.000     1.000      0.709


In [48]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

conds = [("CORN", "runs/modernbert_corn"), ("Nominal", "runs/modernbert_nominal")]

fig = make_subplots(rows=1, cols=2, subplot_titles=[c[0] for c in conds],
                    shared_yaxes=True, shared_xaxes=True)

for j, (name, run_dir) in enumerate(conds, start=1):
    d = load_preds(run_dir)
    for lab, lname, col in [(1, "Correct", "rgba(70,110,170,0.6)"),
                            (0, "Wrong", "rgba(200,90,80,0.6)")]:
        fig.add_histogram(
            x=d.loc[d["correct"] == lab, "p_pred"], name=lname,
            marker_color=col, xbins=dict(start=0, end=1, size=0.025),
            histnorm="probability", legendgroup=lname, showlegend=(j == 1),
            row=1, col=j,
        )

fig.update_layout(
    barmode="overlay",
    title=dict(text="Confidence in predicted label, by outcome"),
    font=dict(family="Arial", size=16, color="black"),
    plot_bgcolor="white", width=980, height=440,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.update_xaxes(title_text="p_pred", range=[0.15, 0.8], gridcolor="rgba(0,0,0,0.1)")
fig.update_yaxes(title_text="Proportion", gridcolor="rgba(0,0,0,0.1)", row=1, col=1)
fig.show()

# Risk-coverage curve

In [50]:
import plotly.graph_objects as go

conds_c = [("CORN", "runs/modernbert_corn", "rgb(70,110,170)"),
           ("Nominal", "runs/modernbert_nominal", "rgb(200,90,80)")]

fig = go.Figure()
for name, run_dir, col in conds_c:
    d = load_preds(run_dir).sort_values("p_pred", ascending=False).reset_index(drop=True)
    d["coverage"] = (np.arange(len(d)) + 1) / len(d)
    d["cum_adj"] = d["adjacent"].expanding().mean()
    d["cum_acc"] = d["correct"].expanding().mean()
    d = d[d["coverage"] >= 0.10]
    fig.add_scatter(x=d["coverage"], y=d["cum_adj"], name=f"{name} — adjacent",
                    line=dict(color=col, width=2))
    fig.add_scatter(x=d["coverage"], y=d["cum_acc"], name=f"{name} — exact",
                    line=dict(color=col, width=2, dash="dash"))

fig.update_layout(
    title=dict(text="Risk–coverage: ModernBERT CORN vs nominal (val)"),
    xaxis_title="Coverage (auto-scored, most confident first)",
    yaxis_title="Accuracy on retained cases",
    font=dict(family="Arial", size=16, color="black"),
    plot_bgcolor="white", width=820, height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.update_xaxes(tickformat=".0%", gridcolor="rgba(0,0,0,0.1)")
fig.update_yaxes(range=[0.3, 1.0], gridcolor="rgba(0,0,0,0.1)")
fig.show()